In [1]:
# Import polars for parquet processing and set Debug level do verbose
import polars as pl
pl.Config.set_verbose(True)

polars.config.Config

In [2]:
# Scan the parquet file, set the following schema to greater reduce memory usage
active_repos_cumulative_stats_by_day = pl.scan_parquet(source='daily_1000_stars.parquet',
                                                       schema={
                                                           "repo_name": pl.String,
                                                           "day": pl.Date,
                                                           "total_stars": pl.UInt32,
                                                           "total_forks": pl.UInt32,
                                                           "total_issues_opened": pl.UInt32,
                                                           "total_issues_closed": pl.UInt32,
                                                           "total_prs_opened": pl.UInt32,
                                                           "total_prs_merged": pl.UInt32,
                                                           "total_commits": pl.UInt32,
                                                           "total_comments": pl.UInt32
                                                       })

_init_credential_provider_builder(): credential_provider_init = None


In [3]:
# distinct_active_repository_names = active_repos_cumulative_stats_by_day.select('repo_name').unique().collect()

In [4]:
# no more sampling (17JAN)
# sampled_repository_names = distinct_active_repository_names.sample(n=1_000_000, with_replacement=False, seed=42)

In [5]:
# Free up memory
# del distinct_active_repository_names

In [6]:
# Keep only sampled repositories (by inner joining the samples with the original dataframe)
# active_sampled_repositories = active_repos_cumulative_stats_by_day.collect().join(other=sampled_repository_names,
#                                                                                   on='repo_name', how='inner',
                                                                                  # maintain_order='left')

In [7]:
# Free up memory
# del sampled_repository_names

In [8]:
# Checkpoint save
# active_sampled_repositories.write_csv('sampled_daily_2M_repos_sorted.csv')

_init_credential_provider_builder(): credential_provider_init = None


In [11]:
# Checkpoint load
# active_sampled_repositories = pl.read_csv('sampled_daily_2M_repos_sorted.csv')

In [3]:
metric_cols = [
    'total_stars', 'total_forks', 'total_issues_opened',
    'total_prs_opened', 'total_commits', 'total_issues_closed', 'total_prs_merged', 'total_comments'
]
# removed:'total_issues_closed', 'total_prs_merged', 'total_comments' (17JAN) 

In [21]:
# Mention set_sorted by repo_name and day, as original data has this property. Boosts memory and cpu optimisation.
active_repos_cumulative_stats_by_day.set_sorted(['repo_name', 'day'])

In [4]:
# exclude these(%_%): (comments not include changes)
# comments
# prs_merged
# lag_1d
repositories_with_rolling_metrics = (active_repos_cumulative_stats_by_day
                                     # Cast to Float (requirement for all subsequent math)
                                     .with_columns([pl.col(c).cast(pl.Float32).alias(c) for c in metric_cols])

                                     # Lags and Rolling Statistics
                                     .with_columns(
    # Lags (1, 7  days) - newreq: remove from all lag/lead cols observations with val = 0
    [pl.col(c).shift(1).over('repo_name').alias(f"{c}_lag_1d") for c in metric_cols] +
    [pl.col(c).shift(7).over('repo_name').alias(f"{c}_lag_7d") for c in metric_cols] +
    [pl.col(c).shift(30).over('repo_name').alias(f"{c}_lag_30d") for c in metric_cols] +
    [pl.col(c).shift(60).over('repo_name').alias(f"{c}_lag_60d") for c in metric_cols] +
    [pl.col(c).shift(-30).over('repo_name').alias(f"{c}_lead_30d") for c in metric_cols] +
    [pl.col(c).shift(-60).over('repo_name').alias(f"{c}_lead_60d") for c in metric_cols] +
    [pl.col(c).shift(-90).over('repo_name').alias(f"{c}_lead_90d") for c in metric_cols] +
    [pl.col(c).shift(-180).over('repo_name').alias(f"{c}_lead_180d") for c in metric_cols] +

    # Growth Rates (Pct Change; 1, 7  days)
    [pl.col(c).pct_change(n=1).over('repo_name').alias(f"{c}_growth_1d") for c in metric_cols] +
    [pl.col(c).pct_change(n=7).over('repo_name').alias(f"{c}_growth_7d") for c in metric_cols] +

    # Rolling Means (7, 30 days)
    [pl.col(c).rolling_mean(window_size=7).over('repo_name').alias(f"{c}_rolling_mean_7d") for c in metric_cols] +
    [pl.col(c).rolling_mean(window_size=30).over('repo_name').alias(f"{c}_rolling_mean_30d") for c in metric_cols] +

    # Rolling Stds (7, 30 days)
    [pl.col(c).rolling_std(window_size=7).over('repo_name').alias(f"{c}_rolling_std_7d") for c in metric_cols] +
    [pl.col(c).rolling_std(window_size=30).over('repo_name').alias(f"{c}_rolling_std_30d") for c in metric_cols]
)

                                     # 4. Net Change (Row-wise math, NO .over needed here as columns are aligned)
                                     .with_columns([
    (pl.col(c) - pl.col(f"{c}_lag_1d")).alias(f"{c}_daily_change")
    for c in metric_cols
])

                                     # 5. Cleanup
                                     .with_columns([
    pl.when(pl.col(pl.Float32).is_infinite())
    .then(None)
    .otherwise(pl.col(pl.Float32))
    .name.keep()
])
                                     .fill_nan(0).fill_null(0)
                                     )
# 30d
# median

In [5]:
#set without any kind of scaling
print("Starting processing... this may take a while.")

(repositories_with_rolling_metrics
    .drop(pl.col("^.*growth.*$"))
    .filter(pl.col("total_stars_lead_180d") != 0)
    .filter(pl.col("total_stars_lag_60d") != 0)
 .sink_parquet('github_features.parquet'))

print("Done! File saved.")

Starting processing... this may take a while.


_init_credential_provider_builder(): credential_provider_init = None


Done! File saved.


In [8]:
# Logarithmic transformation to stabilize variance and reduce data skewness.
repositories_with_log1p_metrics = repositories_with_rolling_metrics.with_columns([
    pl.col(c).log1p().alias(f"{c}_log1p")
    for c in repositories_with_rolling_metrics.columns if
    ("growth" not in c) and ("slope" not in c) and ("day" not in c) and ("repo_name" not in c)
])

C:\Users\vali_\AppData\Local\Temp\ipykernel_7308\22756267.py:4: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  for c in repositories_with_rolling_metrics.columns if


In [ ]:
repositories_with_relevant_metrics_onlylog1pd = repositories_with_log1p_metrics.select(
    pl.col("repo_name"),
    pl.col("day"),
    pl.col(metric_cols),
    pl.col("^.*log1p$")
)
# remove lag1d in selection
repositories_with_relevant_metrics_onlylog1pd.drop(pl.col("^.*lag_1d$"))

In [18]:
repositories_with_relevant_metrics_onlylog1pd.head()

repo_name,day,total_stars,total_forks,total_issues_opened,total_issues_closed,total_prs_opened,total_prs_merged,total_commits,total_comments,total_stars_log1p,total_forks_log1p,total_issues_opened_log1p,total_issues_closed_log1p,total_prs_opened_log1p,total_prs_merged_log1p,total_commits_log1p,total_comments_log1p,total_stars_lag_1d_log1p,total_forks_lag_1d_log1p,total_issues_opened_lag_1d_log1p,total_issues_closed_lag_1d_log1p,total_prs_opened_lag_1d_log1p,total_prs_merged_lag_1d_log1p,total_commits_lag_1d_log1p,total_comments_lag_1d_log1p,total_stars_lag_7d_log1p,total_forks_lag_7d_log1p,total_issues_opened_lag_7d_log1p,total_issues_closed_lag_7d_log1p,total_prs_opened_lag_7d_log1p,total_prs_merged_lag_7d_log1p,total_commits_lag_7d_log1p,total_comments_lag_7d_log1p,total_stars_rolling_mean_7d_log1p,total_forks_rolling_mean_7d_log1p,total_issues_opened_rolling_mean_7d_log1p,…,total_issues_closed_rolling_median_7d_log1p,total_prs_opened_rolling_median_7d_log1p,total_prs_merged_rolling_median_7d_log1p,total_commits_rolling_median_7d_log1p,total_comments_rolling_median_7d_log1p,total_stars_rolling_median_30d_log1p,total_forks_rolling_median_30d_log1p,total_issues_opened_rolling_median_30d_log1p,total_issues_closed_rolling_median_30d_log1p,total_prs_opened_rolling_median_30d_log1p,total_prs_merged_rolling_median_30d_log1p,total_commits_rolling_median_30d_log1p,total_comments_rolling_median_30d_log1p,total_stars_rolling_std_7d_log1p,total_forks_rolling_std_7d_log1p,total_issues_opened_rolling_std_7d_log1p,total_issues_closed_rolling_std_7d_log1p,total_prs_opened_rolling_std_7d_log1p,total_prs_merged_rolling_std_7d_log1p,total_commits_rolling_std_7d_log1p,total_comments_rolling_std_7d_log1p,total_stars_rolling_std_30d_log1p,total_forks_rolling_std_30d_log1p,total_issues_opened_rolling_std_30d_log1p,total_issues_closed_rolling_std_30d_log1p,total_prs_opened_rolling_std_30d_log1p,total_prs_merged_rolling_std_30d_log1p,total_commits_rolling_std_30d_log1p,total_comments_rolling_std_30d_log1p,total_stars_daily_change_log1p,total_forks_daily_change_log1p,total_issues_opened_daily_change_log1p,total_issues_closed_daily_change_log1p,total_prs_opened_daily_change_log1p,total_prs_merged_daily_change_log1p,total_commits_daily_change_log1p,total_comments_daily_change_log1p
str,str,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,…,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32
"""/AndroidAsync""","""2013-03-24""",1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.693147,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""/AndroidIntelliJStarter""","""2013-03-18""",1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.693147,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""/AssetManager""","""2013-02-27""",1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.693147,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""/Bookstore""","""2013-03-07""",1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.693147,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""/CSSOM""","""2013-03-10""",1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.693147

In [11]:
print("Starting processing... this may take a while.")

repositories_with_relevant_metrics_onlylog1pd.sink_parquet('github_features_log1pd.parquet')

print("Done! File saved.")

Starting processing... this may take a while.


_init_credential_provider_builder(): credential_provider_init = None


Done! File saved.


In [24]:
demo_read = pl.scan_parquet('github_features.parquet');

_init_credential_provider_builder(): credential_provider_init = None


In [25]:
demo_read.head(100).collect().to_pandas()

,repo_name,day,total_stars,total_forks,total_issues_opened,total_issues_closed,total_prs_opened,total_prs_merged,total_commits,total_comments,...,total_stars_rolling_std_30d,total_forks_rolling_std_30d,total_issues_opened_rolling_std_30d,total_prs_opened_rolling_std_30d,total_commits_rolling_std_30d,total_stars_daily_change,total_forks_daily_change,total_issues_opened_daily_change,total_prs_opened_daily_change,total_commits_daily_change
0,/,2011-04-13,4108.0,2203.0,3591.0,1446,571.0,0,376933.0,2511,...,592.964722,263.356750,291.152344,62.999035,51138.324219,65.0,38.0,21.0,4.0,5630.0
1,/,2011-04-14,4180.0,2233.0,3761.0,1518,575.0,0,381164.0,2681,...,597.638245,264.662567,307.382385,62.181458,50057.687500,72.0,30.0,170.0,4.0,4231.0
2,/,2011-04-15,4245.0,2257.0,5000.0,1911,580.0,0,385930.0,5000,...,601.579163,265.917358,455.798096,61.368717,49103.101562,65.0,24.0,1239.0,5.0,4766.0
3,/,2011-04-16,4280.0,2273.0,5012.0,1929,581.0,0,389266.0,5118,...,603.304382,266.489655,558.039124,60.498466,48220.734375,35.0,16.0,12.0,1.0,3336.0
4,/,2011-04-17,4315.0,2287.0,5035.0,1946,586.0,0,393319.0,5240,...,603.888733,266.469421,637.044006,59.808735,47304.125000,35.0,14.0,23.0,5.0,4053.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,/,2013-01-08,5452.0,2942.0,11466.0,5410,736.0,0,547309.0,17025,...,0.000000,0.000047,2.921187,0.000015,7.991094,0.0,0.0,0.0,0.0,14.0
96,/,2013-01-09,5452.0,2942.0,11467.0,5410,736.0,0,547310.0,17028,...,0.000000,0.000047,2.914293,0.000015,9.740371,0.0,0.0,1.0,0.0,1.0
97,/,2013-01-11,5452.0,2942.0,11467.0,5410,736.0,0,547310.0,17030,...,0.000000,0.000047,2.887946,0.000015,11.093961,0.0,0.0,0.0,0.0,0.0
98,/,2013-01-13,5452.0,2942.0,11467.0,5410,736.0,0,547312.0,17030,...,0.000000,0.000047,2.841604,0.000015,12.344159,0.0,0.0,0.0,0.0,2.0


In [11]:
# Free up memory
del repositories_with_rolling_metrics

In [12]:
cols_to_scale = [c for c in repositories_with_log1p_metrics.columns if c.endswith('_log1p')]

repositories_with_scaled_rolling_metrics = repositories_with_log1p_metrics.with_columns([
    (
        #maxabsscaler
            pl.col(c) / (pl.col(c).max() + 1e-6)
    ).alias(c.replace('_log1p', '_scaled'))
    for c in cols_to_scale
])

In [13]:
del repositories_with_log1p_metrics

In [14]:
repositories_with_relevant_metrics = repositories_with_scaled_rolling_metrics.select(
    pl.col("repo_name"),
    pl.col("day"),
    pl.col(metric_cols),
    pl.col("^.*_scaled$")
)

In [ ]:
print("Starting processing... this may take a while.")

repositories_with_relevant_metrics.write_csv(
    "processed_github_features.csv",
    # maintain_order=False
)
print("Done! File saved.")

In [ ]:
print("Starting processing... this may take a while.")

repositories_with_relevant_metrics.write_parquet('processed_github_features.parquet')

print("Done! File saved.")